# Predicting Crop Performance from Soil Microbial Communities

**AUTHOR**  
Rodrigo Kang

## Overview

This notebook accompanies the project **Predicting Crop Performance from Soil Microbial Communities** and contains the computational implementation supporting the analysis of potato yield, soil chemistry, management information, and pre-plant soil microbial communities.

The workflow focuses on leakage-safe grouped validation, comparison of incremental information blocks, regression model development, and cautious interpretation of predictive associations.

## Imports and Configuration

This section includes the libraries, reproducibility settings, path configuration, output folders, and lightweight helper functions required throughout the notebook.

The notebook is expected to run from the local `python/` directory, load `PotatoSCMP.csv` from the sibling `../data/` directory, and save generated artifacts under `output/`.

In [1]:
# Standard library
import random
import warnings
from pathlib import Path

# Data handling and visualisation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Scikit-learn utilities used throughout the modelling workflow
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)

# Notebook behaviour
warnings.filterwarnings("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

plt.rcParams.update(
    {
        "figure.figsize": (9, 5.5),
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.constrained_layout.use": True,
    }
)

# Relative project paths: the notebook is executed from the local python/ directory.
project_dir = Path.cwd()
data_dir = project_dir.parent / "data"
data_path = data_dir / "PotatoSCMP.csv"
output_dir = project_dir / "output"
figures_dir = output_dir / "figures"
tables_dir = output_dir / "tables"

for directory in (output_dir, figures_dir, tables_dir):
    directory.mkdir(parents=True, exist_ok=True)


def save_figure(figure, filename, close=True):
    """
    Description:
    ------------
    Save a Matplotlib figure as a PNG file in the configured figures folder.

    Input:
    ------
    figure : matplotlib.figure.Figure
        Figure object to save.
    filename : str
        Concise kebab-case filename, with or without the .png suffix.
    close : bool
        Whether to close the figure after saving.

    Outputs:
    --------
    figure_path : pathlib.Path
        Path of the saved PNG file.

    Author:
    -------
    Rodrigo Kang
    """
    figure_path = figures_dir / f"{Path(filename).stem}.png"
    figure.savefig(figure_path, bbox_inches="tight")

    if close:
        plt.close(figure)

    return figure_path


def save_table(table, filename, index=False):
    """
    Description:
    ------------
    Save a pandas table as a CSV file in the configured tables folder.

    Input:
    ------
    table : pandas.DataFrame or pandas.Series
        Table to export.
    filename : str
        Concise kebab-case filename, with or without the .csv suffix.
    index : bool
        Whether to include the pandas index in the exported file.

    Outputs:
    --------
    table_path : pathlib.Path
        Path of the saved CSV file.

    Author:
    -------
    Rodrigo Kang
    """
    table_path = tables_dir / f"{Path(filename).stem}.csv"
    table.to_csv(table_path, index=index)
    return table_path


configuration = pd.Series(
    {
        "project_dir": project_dir,
        "data_path": data_path,
        "output_dir": output_dir,
        "figures_dir": figures_dir,
        "tables_dir": tables_dir,
        "random_seed": random_seed,
    },
    name="value",
)

configuration.to_frame()


,value
project_dir,/mnt/data/implementation-part5/run-check/python
data_path,/mnt/data/implementation-part5/run-check/data/...
output_dir,/mnt/data/implementation-part5/run-check/pytho...
figures_dir,/mnt/data/implementation-part5/run-check/pytho...
tables_dir,/mnt/data/implementation-part5/run-check/pytho...
random_seed,42


## Data Loading

This section loads `PotatoSCMP.csv`, validates its basic structure, and documents the dataset source and provisional manuscript notice. The scientific outcome is `Yield_per_meter`, measured as grams of fresh potato tuber per metre of row. Grouping and modelling decisions are deferred to the Methodology section.

In [2]:
# Dataset source and scientific-use notice
# Dataset page: https://agdatacommons.nal.usda.gov/articles/dataset/29944829
# The associated manuscript was under peer review when the dataset was posted.
# Results should therefore be interpreted as predictive evidence rather than definitive biological conclusions.

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_path}. Run the notebook from the python/ directory "
        "with PotatoSCMP.csv stored in the sibling data/ directory."
    )

raw_data = pd.read_csv(data_path)

expected_columns = {
    "Sample",
    "Link_ID",
    "project",
    "State",
    "Field1",
    "Field2",
    "Plot",
    "Year",
    "Yield_per_meter",
}
missing_expected_columns = sorted(expected_columns.difference(raw_data.columns))

if missing_expected_columns:
    raise ValueError(
        "The dataset is missing required columns: "
        + ", ".join(missing_expected_columns)
    )

if raw_data.shape[0] != 423:
    warnings.warn(
        f"Expected 423 observations from the published dataset, found {raw_data.shape[0]}.",
        stacklevel=2,
    )

if not pd.api.types.is_numeric_dtype(raw_data["Yield_per_meter"]):
    raise TypeError("Yield_per_meter must be numeric for regression modelling.")

if raw_data["Yield_per_meter"].isna().any():
    raise ValueError("Yield_per_meter contains missing values.")

identifier_columns = [
    "Sample",
    "Link_ID",
    "project",
    "State",
    "Field1",
    "Field2",
    "Plot",
    "Year",
]

# State and Field1 identify the physical field more conservatively than Field2,
# which contains a year suffix and therefore represents a field-season.
raw_data["field_group"] = (
    raw_data["State"].astype("string").str.strip()
    + "-"
    + raw_data["Field1"].astype("string").str.strip()
)

dataset_validation = pd.DataFrame(
    {
        "check": [
            "observations",
            "source-columns",
            "analysis-columns",
            "duplicate-rows",
            "duplicate-sample-ids",
            "duplicate-link-ids",
            "missing-target-values",
            "non-positive-target-values",
            "physical-field-groups",
            "field-season-groups",
        ],
        "value": [
            raw_data.shape[0],
            raw_data.shape[1] - 1,
            raw_data.shape[1],
            raw_data.drop(columns="field_group").duplicated().sum(),
            raw_data["Sample"].duplicated().sum(),
            raw_data["Link_ID"].duplicated().sum(),
            raw_data["Yield_per_meter"].isna().sum(),
            raw_data["Yield_per_meter"].le(0).sum(),
            raw_data["field_group"].nunique(),
            raw_data["Field2"].nunique(),
        ],
    }
)

save_table(dataset_validation, "dataset-validation")
dataset_validation


,check,value
0,observations,423
1,source-columns,1631
2,analysis-columns,1632
3,duplicate-rows,0
4,duplicate-sample-ids,0
5,duplicate-link-ids,0
6,missing-target-values,0
7,non-positive-target-values,0
8,physical-field-groups,77
9,field-season-groups,137


## Exploratory Analysis (Optional)

This section provides a descriptive view of the response, conventional predictors, microbial summaries, and taxonomic matrices. These summaries are used to understand scale, missingness, and sparsity only; they do not select features or estimate any preprocessing parameters used in cross-validation.

### Yield Distribution

The response is examined on its original scale. The histogram and boxplot show the overall distribution of fresh tuber yield, while the exported table records robust quantiles for later reference.

In [3]:
target_summary = (
    raw_data["Yield_per_meter"]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
    .rename("grams-per-metre")
    .to_frame()
    .rename_axis("statistic")
    .reset_index()
)

target_summary["statistic"] = target_summary["statistic"].replace(
    {"count": "observations", "std": "standard-deviation"}
)

figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(raw_data["Yield_per_meter"], bins=24, edgecolor="white")
axes[0].axvline(raw_data["Yield_per_meter"].median(), linestyle="--", linewidth=1.5)
axes[0].set(
    title="Yield distribution",
    xlabel="Fresh tuber yield (g per metre)",
    ylabel="Observations",
)
axes[1].boxplot(raw_data["Yield_per_meter"], vert=False)
axes[1].set(
    title="Yield spread",
    xlabel="Fresh tuber yield (g per metre)",
    yticks=[],
)

save_figure(figure, "yield-distribution")
save_table(target_summary, "target-summary")
target_summary

/tmp/ipykernel_8532/3348929786.py:22: PendingDeprecationWarning: vert: bool will be deprecated in a future version. Use orientation: {'vertical', 'horizontal'} instead.
  axes[1].boxplot(raw_data["Yield_per_meter"], vert=False)


,statistic,grams-per-metre
0,observations,423.0000
1,mean,"3,774.9173"
2,standard-deviation,"1,576.7086"
3,min,159.4400
4,5%,"1,589.4000"
5,25%,"2,650.5000"
6,50%,"3,588.9528"
7,75%,"4,618.6250"
8,95%,"6,563.6000"
9,max,"10,977.7794"


### Missingness and Conventional Variables

Missingness is summarised before any imputation. Soil chemistry is explored through its pairwise correlation structure, which is descriptive and is not used to remove variables outside the training folds.

In [4]:
eda_soil_columns = [
    "pH_1_1", "CEC", "OM_percent", "P_ppm", "K_ppm", "Mg_ppm", "Ca_ppm",
    "K_Sat_percent", "Mg_Sat_percent", "Ca_Sat_percent",
]

missingness_summary = (
    raw_data.drop(columns="field_group")
    .isna()
    .sum()
    .rename("missing-values")
    .to_frame()
    .assign(
        **{
            "missing-percent": lambda frame: (
                100 * frame["missing-values"] / len(raw_data)
            )
        }
    )
    .query("`missing-values` > 0")
    .sort_values(["missing-values"], ascending=False)
    .rename_axis("variable")
    .reset_index()
)

soil_correlation = raw_data[eda_soil_columns].corr()

figure, axis = plt.subplots(figsize=(8.5, 7))
image = axis.imshow(soil_correlation, vmin=-1, vmax=1, aspect="auto")
axis.set_xticks(range(len(eda_soil_columns)), eda_soil_columns, rotation=55, ha="right")
axis.set_yticks(range(len(eda_soil_columns)), eda_soil_columns)
axis.set_title("Soil chemistry correlations")
figure.colorbar(image, ax=axis, label="Pearson correlation")

save_figure(figure, "soil-correlation")
save_table(missingness_summary, "missingness-summary")
save_table(
    soil_correlation.rename_axis("variable").reset_index(),
    "soil-correlation",
)

display(missingness_summary)
soil_correlation.round(2)

,variable,missing-values,missing-percent
0,Rotation length,72,17.0213
1,Rotation diversity,72,17.0213
2,B16S_pcoa1,66,15.6028
3,B16S_pcoa2,66,15.6028
4,ITS_pcoa1,66,15.6028
5,ITS_pcoa2,66,15.6028


,pH_1_1,CEC,OM_percent,P_ppm,K_ppm,Mg_ppm,Ca_ppm,K_Sat_percent,Mg_Sat_percent,Ca_Sat_percent
pH_1_1,1.0000,-0.1000,-0.1500,-0.4300,0.0400,0.6100,0.3600,0.0900,0.7900,0.8300
CEC,-0.1000,1.0000,0.7600,-0.1000,0.4500,0.5500,0.8200,-0.1000,-0.2900,-0.0300
OM_percent,-0.1500,0.7600,1.0000,-0.1300,0.2600,0.4300,0.6800,-0.0800,-0.2600,0.0100
P_ppm,-0.4300,-0.1000,-0.1300,1.0000,0.0300,-0.3600,-0.3000,0.0600,-0.3400,-0.4200
K_ppm,0.0400,0.4500,0.2600,0.0300,1.0000,0.2000,0.3100,0.7800,-0.1800,-0.1300
Mg_ppm,0.6100,0.5500,0.4300,-0.3600,0.2000,1.0000,0.7400,-0.1000,0.5900,0.4800
Ca_ppm,0.3600,0.8200,0.6800,-0.3000,0.3100,0.7400,1.0000,-0.1500,0.0400,0.5100
K_Sat_percent,0.0900,-0.1000,-0.0800,0.0600,0.7800,-0.1000,-0.1500,1.0000,-0.0300,-0.1500
Mg_Sat_percent,0.7900,-0.2900,-0.2600,-0.3400,-0.1800,0.5900,0.0400,-0.0300,1.0000,0.5600
Ca_Sat_percent,0.8300,-0.0300,0.0100,-0.4200,-0.1300,0.4800,0.5100,-0.1500,0.5600,1.0000


### Microbial Summary Variables

qPCR abundance variables and Shannon diversity indices are kept distinct because they represent different biological summaries. Their distributions are shown without standardisation or outcome-driven screening.

In [5]:
eda_qpcr_columns = ["logV", "logS", "log16S", "logITS"]
eda_shannon_columns = [
    "Bac_G_Shannon", "Fun_G_Shannon", "Bac_O_Shannon",
    "Fun_O_Shannon", "Bac_P_Shannon", "Fun_P_Shannon",
]
eda_microbial_columns = eda_qpcr_columns + eda_shannon_columns

microbial_summary = (
    raw_data[eda_microbial_columns]
    .describe(percentiles=[0.05, 0.50, 0.95])
    .transpose()
    .rename_axis("variable")
    .reset_index()
)
microbial_summary["information-type"] = microbial_summary["variable"].map(
    lambda variable: "qpcr-abundance" if variable in eda_qpcr_columns else "shannon-diversity"
)

figure, axes = plt.subplots(2, 5, figsize=(15, 7.5))
for axis, column in zip(axes.flat, eda_microbial_columns):
    axis.hist(raw_data[column].dropna(), bins=18, edgecolor="white")
    axis.set_title(column)
    axis.set_ylabel("Observations")
for axis in axes.flat[len(eda_microbial_columns):]:
    axis.set_visible(False)
figure.suptitle("Microbial abundance and diversity summaries", fontsize=14)

save_figure(figure, "microbial-summaries")
save_table(microbial_summary, "microbial-summary")
microbial_summary

,variable,count,mean,std,min,5%,50%,95%,max,information-type
0,logV,423.0000,0.0809,0.9555,-3.2095,-1.4166,0.0777,1.6662,2.9594,qpcr-abundance
1,logS,423.0000,0.1502,0.9558,-2.8851,-1.3390,0.1950,1.6994,3.4707,qpcr-abundance
2,log16S,423.0000,0.0011,1.0300,-2.9733,-1.5415,-0.0531,1.8857,3.0325,qpcr-abundance
3,logITS,423.0000,-0.0154,0.9816,-2.8501,-1.5927,-0.0041,1.6160,4.2229,qpcr-abundance
4,Bac_G_Shannon,423.0000,-0.0025,0.9547,-4.0000,-1.9703,0.1469,1.2356,2.0132,shannon-diversity
5,Fun_G_Shannon,423.0000,0.0704,0.9151,-6.6962,-1.3242,0.2246,1.1951,1.8628,shannon-diversity
6,Bac_O_Shannon,423.0000,0.0557,0.9653,-3.4095,-1.8255,0.1341,1.4861,2.0441,shannon-diversity
7,Fun_O_Shannon,423.0000,0.0386,0.9596,-5.7154,-1.7167,0.2407,1.1533,1.5257,shannon-diversity
8,Bac_P_Shannon,423.0000,-0.0033,1.0401,-3.3656,-2.2424,0.0809,1.4361,1.9649,shannon-diversity
9,Fun_P_Shannon,423.0000,0.0176,0.9744,-3.7298,-1.6885,0.2538,1.2520,2.1751,shannon-diversity


### Taxonomic Sparsity and Prevalence

Bacterial and fungal relative-abundance matrices are evaluated separately. The prevalence curves show how often each genus is observed and make the high-dimensional, zero-heavy structure explicit. They do not determine the final retained taxa, because prevalence filtering is learned again inside each training fold.

In [6]:
eda_bacterial_columns = [
    column for column in raw_data.columns if column.startswith("BF_g_")
]
eda_fungal_columns = [
    column for column in raw_data.columns if column.startswith("FF_g_")
]

bacterial_prevalence = (raw_data[eda_bacterial_columns] > 0).mean().sort_values(ascending=False)
fungal_prevalence = (raw_data[eda_fungal_columns] > 0).mean().sort_values(ascending=False)

taxa_eda_summary = pd.DataFrame(
    {
        "community": ["bacterial", "fungal"],
        "taxa": [len(eda_bacterial_columns), len(eda_fungal_columns)],
        "zero-percent": [
            100 * raw_data[eda_bacterial_columns].eq(0).to_numpy().mean(),
            100 * raw_data[eda_fungal_columns].eq(0).to_numpy().mean(),
        ],
        "median-prevalence": [bacterial_prevalence.median(), fungal_prevalence.median()],
        "taxa-prevalence-10pct": [
            bacterial_prevalence.ge(0.10).sum(),
            fungal_prevalence.ge(0.10).sum(),
        ],
        "median-row-sum": [
            raw_data[eda_bacterial_columns].sum(axis=1).median(),
            raw_data[eda_fungal_columns].sum(axis=1).median(),
        ],
    }
)

figure, axis = plt.subplots(figsize=(9, 5.5))
axis.plot(np.arange(1, len(bacterial_prevalence) + 1), bacterial_prevalence.to_numpy(), label="Bacteria")
axis.plot(np.arange(1, len(fungal_prevalence) + 1), fungal_prevalence.to_numpy(), label="Fungi")
axis.axhline(0.10, linestyle="--", linewidth=1.25, label="10% reference")
axis.set(
    title="Taxon prevalence by ranked genus",
    xlabel="Genus rank",
    ylabel="Proportion of samples with non-zero abundance",
    ylim=(0, 1.02),
)
axis.legend()

save_figure(figure, "taxa-prevalence")
save_table(taxa_eda_summary, "taxa-eda-summary")
taxa_eda_summary

,community,taxa,zero-percent,median-prevalence,taxa-prevalence-10pct,median-row-sum
0,bacterial,885,70.8814,0.1371,497,0.9064
1,fungal,706,80.1666,0.0662,300,0.8874


### Exploratory Takeaways

Yield remains a continuous outcome with substantial between-sample variation. Missing values are concentrated in a small number of conventional variables and therefore require fold-specific imputation. Soil chemistry contains correlated measurements, while qPCR and Shannon variables occupy different numerical scales. The bacterial and fungal matrices are both sparse, with fungal taxa showing greater zero inflation. These observations motivate leakage-safe preprocessing but do not establish causal biological relationships or justify conclusions about microbial mechanisms.

## Methodology

This section defines the leakage-safe experimental framework used to test whether pre-plant soil microbial information improves potato-yield prediction beyond conventional management and soil chemistry variables. It covers field grouping, information blocks, grouped cross-validation, and preprocessing learned exclusively within training folds.

### Field Group Reconstruction

The validation unit is reconstructed before creating folds. The published description refers to 130 commercial fields, but the supplied identifiers do not reproduce that count exactly. `Field2` yields 137 field-season labels, whereas `State + Field1` yields 77 physical-field groups. The latter is selected as a conservative grouping rule because it keeps repeated observations from the same physical field together across years and projects. This choice is documented as a limitation and can later be tested in a sensitivity analysis.

In [7]:
grouping_review = pd.DataFrame(
    {
        "candidate": [
            "link-id",
            "project-state-field2",
            "state-field2",
            "project-state-field1",
            "state-field1",
        ],
        "interpretation": [
            "sample-level identifier",
            "project-specific field-season",
            "field-season across projects",
            "project-specific physical field",
            "physical field across projects and years",
        ],
        "n_groups": [
            raw_data["Link_ID"].nunique(),
            raw_data[["project", "State", "Field2"]].drop_duplicates().shape[0],
            raw_data[["State", "Field2"]].drop_duplicates().shape[0],
            raw_data[["project", "State", "Field1"]].drop_duplicates().shape[0],
            raw_data["field_group"].nunique(),
        ],
        "selected_for_validation": [False, False, False, False, True],
    }
)

field_group_summary = (
    raw_data.groupby("field_group", as_index=False)
    .agg(
        observations=("Link_ID", "size"),
        projects=("project", "nunique"),
        seasons=("Year", "nunique"),
        field_season_labels=("Field2", "nunique"),
    )
    .sort_values(["observations", "field_group"], ascending=[False, True])
    .reset_index(drop=True)
)

identifier_missingness = (
    raw_data[identifier_columns + ["field_group"]]
    .isna()
    .sum()
    .rename("missing_values")
    .rename_axis("identifier").reset_index()
)

save_table(grouping_review, "field-grouping-review")
save_table(field_group_summary, "field-group-summary")
save_table(identifier_missingness, "identifier-missingness")

display(grouping_review)
display(field_group_summary.head(10))


,candidate,interpretation,n_groups,selected_for_validation
0,link-id,sample-level identifier,423,False
1,project-state-field2,project-specific field-season,143,False
2,state-field2,field-season across projects,137,False
3,project-state-field1,project-specific physical field,89,False
4,state-field1,physical field across projects and years,77,True


,field_group,observations,projects,seasons,field_season_labels
0,WI-5,17,2,4,4
1,WI-6,16,2,4,4
2,WI-4,15,2,4,4
3,WI-8,13,2,4,4
4,WI-1,12,2,3,3
5,WI-2,12,2,3,3
6,WI-26,12,2,3,3
7,WI-3,12,2,3,3
8,WI-7,11,2,3,3
9,WI-25,10,2,3,3


### Information Blocks

Variables are assigned to scientifically meaningful information blocks before model fitting. Identifiers are excluded, microbial summaries are distinguished from full taxonomic composition, and PCoA axes are not combined with the complete abundance matrices in the same feature set unless explicitly stated.

In [8]:
target_column = "Yield_per_meter"
group_column = "field_group"

identifier_columns = [
    "Sample",
    "Link_ID",
    "Field1",
    "Field2",
    "Plot",
]

management_columns = [
    "project",
    "State",
    "Year",
    "Variety1",
    "Variety2",
    "Fumigation1",
    "Region",
    "Soil_type",
    "Rotation length",
    "Rotation diversity",
]

soil_chemistry_columns = [
    "pH_1_1",
    "CEC",
    "OM_percent",
    "P_ppm",
    "K_ppm",
    "Mg_ppm",
    "Ca_ppm",
    "K_Sat_percent",
    "Mg_Sat_percent",
    "Ca_Sat_percent",
]

qpcr_columns = ["logV", "logS", "log16S", "logITS"]

shannon_columns = [
    "Bac_G_Shannon",
    "Fun_G_Shannon",
    "Bac_O_Shannon",
    "Fun_O_Shannon",
    "Bac_P_Shannon",
    "Fun_P_Shannon",
]

pcoa_columns = [
    "B16S_pcoa1",
    "B16S_pcoa2",
    "ITS_pcoa1",
    "ITS_pcoa2",
]

bacterial_taxa_columns = [
    column for column in raw_data.columns if column.startswith("BF_g_")
]
fungal_taxa_columns = [
    column for column in raw_data.columns if column.startswith("FF_g_")
]

feature_blocks = {
    "management": management_columns,
    "soil-chemistry": soil_chemistry_columns,
    "qpcr-abundance": qpcr_columns,
    "shannon-diversity": shannon_columns,
    "pcoa-axes": pcoa_columns,
    "bacterial-taxa": bacterial_taxa_columns,
    "fungal-taxa": fungal_taxa_columns,
}


def validate_feature_partition(data, blocks, excluded_columns):
    """
    Description:
    ------------
    Validate that feature blocks exist, do not overlap, and cover all source columns.

    Input:
    ------
    data : pandas.DataFrame
        Dataset containing the source variables.
    blocks : dict
        Mapping from feature-block names to column lists.
    excluded_columns : list
        Columns intentionally excluded from the predictive feature blocks.

    Outputs:
    --------
    partition_summary : pandas.DataFrame
        Validation summary for feature coverage and overlap.

    Author:
    -------
    Rodrigo Kang
    """
    block_columns = [column for columns in blocks.values() for column in columns]
    duplicate_columns = sorted(
        pd.Series(block_columns)[pd.Series(block_columns).duplicated()].unique()
    )
    missing_columns = sorted(set(block_columns).difference(data.columns))
    uncovered_columns = sorted(
        set(data.columns).difference(block_columns).difference(excluded_columns)
    )

    partition_summary = pd.DataFrame(
        {
            "check": [
                "block-columns",
                "duplicate-block-columns",
                "missing-block-columns",
                "uncovered-source-columns",
            ],
            "value": [
                len(block_columns),
                len(duplicate_columns),
                len(missing_columns),
                len(uncovered_columns),
            ],
            "details": [
                "",
                ", ".join(duplicate_columns),
                ", ".join(missing_columns),
                ", ".join(uncovered_columns),
            ],
        }
    )

    if duplicate_columns or missing_columns or uncovered_columns:
        raise ValueError(
            "Feature partition is incomplete or overlapping. Review partition_summary."
        )

    return partition_summary


excluded_columns = identifier_columns + [target_column, group_column]
partition_summary = validate_feature_partition(
    raw_data,
    feature_blocks,
    excluded_columns,
)

column_to_block = {
    column: block_name
    for block_name, columns in feature_blocks.items()
    for column in columns
}

feature_audit = pd.DataFrame(
    {
        "column": [column for columns in feature_blocks.values() for column in columns]
    }
)
feature_audit["block"] = feature_audit["column"].map(column_to_block)
feature_audit["dtype"] = feature_audit["column"].map(
    raw_data.dtypes.astype(str).to_dict()
)
feature_audit["missing-count"] = feature_audit["column"].map(
    raw_data.isna().sum().to_dict()
)
feature_audit["missing-rate"] = feature_audit["missing-count"] / len(raw_data)
feature_audit["unique-values"] = feature_audit["column"].map(
    raw_data.nunique(dropna=True).to_dict()
)
feature_audit["constant"] = feature_audit["unique-values"].le(1)

numeric_columns = feature_audit.loc[
    feature_audit["dtype"].ne("object"), "column"
].tolist()
zero_rate = raw_data[numeric_columns].eq(0).mean().to_dict()
feature_audit["zero-rate"] = feature_audit["column"].map(zero_rate)

feature_block_summary = (
    feature_audit.groupby("block", sort=False)
    .agg(
        **{
            "features": ("column", "size"),
            "numeric-features": (
                "dtype",
                lambda values: values.ne("object").sum(),
            ),
            "categorical-features": (
                "dtype",
                lambda values: values.eq("object").sum(),
            ),
            "features-with-missing": (
                "missing-count",
                lambda values: values.gt(0).sum(),
            ),
            "mean-missing-rate": ("missing-rate", "mean"),
            "mean-zero-rate": ("zero-rate", "mean"),
            "constant-features": ("constant", "sum"),
        }
    )
    .reset_index()
)

composition_summary = pd.DataFrame(
    {
        "community": ["bacterial", "fungal"],
        "features": [len(bacterial_taxa_columns), len(fungal_taxa_columns)],
        "mean-zero-rate": [
            raw_data[bacterial_taxa_columns].eq(0).mean().mean(),
            raw_data[fungal_taxa_columns].eq(0).mean().mean(),
        ],
        "median-sample-sum": [
            raw_data[bacterial_taxa_columns].sum(axis=1).median(),
            raw_data[fungal_taxa_columns].sum(axis=1).median(),
        ],
        "minimum-sample-sum": [
            raw_data[bacterial_taxa_columns].sum(axis=1).min(),
            raw_data[fungal_taxa_columns].sum(axis=1).min(),
        ],
        "maximum-sample-sum": [
            raw_data[bacterial_taxa_columns].sum(axis=1).max(),
            raw_data[fungal_taxa_columns].sum(axis=1).max(),
        ],
    }
)

# These sets answer incremental scientific questions rather than benchmarking every
# possible combination. PCoA axes are not combined with full taxonomic compositions.
feature_sets = {
    "conventional": management_columns + soil_chemistry_columns,
    "microbial-summaries": qpcr_columns + shannon_columns + pcoa_columns,
    "conventional-plus-summaries": (
        management_columns
        + soil_chemistry_columns
        + qpcr_columns
        + shannon_columns
        + pcoa_columns
    ),
    "taxonomic-composition": bacterial_taxa_columns + fungal_taxa_columns,
    "conventional-plus-taxa": (
        management_columns
        + soil_chemistry_columns
        + bacterial_taxa_columns
        + fungal_taxa_columns
    ),
    "integrated": (
        management_columns
        + soil_chemistry_columns
        + qpcr_columns
        + shannon_columns
        + bacterial_taxa_columns
        + fungal_taxa_columns
    ),
}

feature_set_summary = pd.DataFrame(
    [
        {
            "feature-set": feature_set_name,
            "features": len(columns),
            "includes-management": bool(set(columns) & set(management_columns)),
            "includes-chemistry": bool(set(columns) & set(soil_chemistry_columns)),
            "includes-qpcr": bool(set(columns) & set(qpcr_columns)),
            "includes-shannon": bool(set(columns) & set(shannon_columns)),
            "includes-pcoa": bool(set(columns) & set(pcoa_columns)),
            "includes-taxa": bool(
                set(columns)
                & set(bacterial_taxa_columns + fungal_taxa_columns)
            ),
        }
        for feature_set_name, columns in feature_sets.items()
    ]
)

save_table(partition_summary, "feature-partition")
save_table(feature_audit, "feature-audit")
save_table(feature_block_summary, "feature-blocks")
save_table(composition_summary, "composition-summary")
save_table(feature_set_summary, "feature-sets")

feature_block_summary


,block,features,numeric-features,categorical-features,features-with-missing,mean-missing-rate,mean-zero-rate,constant-features
0,management,10,2,8,2,0.0340,0.0000,0
1,soil-chemistry,10,10,0,0,0.0000,0.0009,0
2,qpcr-abundance,4,4,0,0,0.0000,0.0000,0
3,shannon-diversity,6,6,0,0,0.0000,0.0000,0
4,pcoa-axes,4,4,0,4,0.1560,0.0000,0
5,bacterial-taxa,885,885,0,0,0.0000,0.7088,0
6,fungal-taxa,706,706,0,0,0.0000,0.8017,0


### Grouped Validation and Fold-Safe Preprocessing

Five fixed `GroupKFold` splits are used for all later comparisons. Numeric imputation, categorical encoding, taxon prevalence filtering, zero replacement, CLR transformation, scaling, and subsequent hyperparameter selection remain inside the training data for each fold.

In [9]:
class PrevalenceCLRTransformer(BaseEstimator, TransformerMixin):
    """
    Description:
    ------------
    Filter sparse taxa using training prevalence and apply a centred log-ratio transform.

    Input:
    ------
    min_prevalence : float
        Minimum positive-value prevalence required to retain a taxon.
    pseudocount_scale : float
        Fraction of the smallest positive training abundance used as pseudocount.

    Outputs:
    --------
    transformer : PrevalenceCLRTransformer
        Scikit-learn-compatible transformer with learned taxa and pseudocount.

    Author:
    -------
    Rodrigo Kang
    """

    def __init__(self, min_prevalence=0.10, pseudocount_scale=0.50):
        """
        Description:
        ------------
        Initialise prevalence and pseudocount settings.

        Input:
        ------
        min_prevalence : float
            Minimum proportion of positive training observations.
        pseudocount_scale : float
            Multiplier applied to the smallest positive training value.

        Outputs:
        --------
        None
            Stores configuration without learning from data.

        Author:
        -------
        Rodrigo Kang
        """
        self.min_prevalence = min_prevalence
        self.pseudocount_scale = pseudocount_scale

    def fit(self, x, y=None):
        """
        Description:
        ------------
        Learn retained taxa and the zero-replacement pseudocount from training data.

        Input:
        ------
        x : array-like
            Non-negative taxonomic relative-abundance matrix.
        y : array-like or None
            Unused target values accepted for pipeline compatibility.

        Outputs:
        --------
        self : PrevalenceCLRTransformer
            Fitted transformer.

        Author:
        -------
        Rodrigo Kang
        """
        x_array = self._validate_input(x)

        if not 0 < self.min_prevalence <= 1:
            raise ValueError("min_prevalence must be in the interval (0, 1].")
        if self.pseudocount_scale <= 0:
            raise ValueError("pseudocount_scale must be positive.")

        prevalence = np.mean(x_array > 0, axis=0)
        self.support_mask_ = prevalence >= self.min_prevalence

        if not np.any(self.support_mask_):
            raise ValueError("No taxa satisfy the requested prevalence threshold.")

        retained_values = x_array[:, self.support_mask_]
        positive_values = retained_values[retained_values > 0]

        if positive_values.size == 0:
            raise ValueError("Retained taxonomic features contain no positive values.")

        self.pseudocount_ = float(positive_values.min() * self.pseudocount_scale)
        self.n_features_in_ = x_array.shape[1]
        self.feature_names_in_ = np.asarray(
            getattr(x, "columns", [f"taxon-{index}" for index in range(x_array.shape[1])]),
            dtype=object,
        )
        return self

    def transform(self, x):
        """
        Description:
        ------------
        Apply training-derived filtering, zero replacement, closure and CLR transformation.

        Input:
        ------
        x : array-like
            Taxonomic matrix with the same source columns used during fitting.

        Outputs:
        --------
        clr_values : numpy.ndarray
            CLR-transformed retained taxonomic features.

        Author:
        -------
        Rodrigo Kang
        """
        if not hasattr(self, "support_mask_"):
            raise RuntimeError("The transformer must be fitted before transform is called.")

        x_array = self._validate_input(x)
        if x_array.shape[1] != self.n_features_in_:
            raise ValueError("Taxonomic input has a different number of columns than training data.")

        retained_values = x_array[:, self.support_mask_]
        replaced_values = np.where(
            retained_values > 0,
            retained_values,
            self.pseudocount_,
        )
        closed_values = replaced_values / replaced_values.sum(axis=1, keepdims=True)
        log_values = np.log(closed_values)
        clr_values = log_values - log_values.mean(axis=1, keepdims=True)
        return clr_values

    def get_feature_names_out(self, input_features=None):
        """
        Description:
        ------------
        Return names of taxa retained during fitting.

        Input:
        ------
        input_features : array-like or None
            Optional original feature names supplied by scikit-learn.

        Outputs:
        --------
        retained_names : numpy.ndarray
            Names of retained taxonomic features.

        Author:
        -------
        Rodrigo Kang
        """
        if not hasattr(self, "support_mask_"):
            raise RuntimeError("The transformer must be fitted before feature names are requested.")

        source_names = (
            np.asarray(input_features, dtype=object)
            if input_features is not None
            else self.feature_names_in_
        )
        return source_names[self.support_mask_]

    @staticmethod
    def _validate_input(x):
        """
        Description:
        ------------
        Convert taxonomic input to a finite, non-negative numeric array.

        Input:
        ------
        x : array-like
            Candidate taxonomic matrix.

        Outputs:
        --------
        x_array : numpy.ndarray
            Validated floating-point matrix.

        Author:
        -------
        Rodrigo Kang
        """
        x_array = np.asarray(x, dtype=float)

        if x_array.ndim != 2:
            raise ValueError("Taxonomic input must be a two-dimensional matrix.")
        if not np.isfinite(x_array).all():
            raise ValueError("Taxonomic input contains missing or infinite values.")
        if (x_array < 0).any():
            raise ValueError("Taxonomic relative abundances must be non-negative.")

        return x_array


def build_preprocessor(feature_set_name, scale_numeric=True):
    """
    Description:
    ------------
    Build a fold-safe preprocessing pipeline for one predefined information set.

    Input:
    ------
    feature_set_name : str
        Name of a feature set defined in feature_sets.
    scale_numeric : bool
        Whether to standardise continuous and CLR-transformed variables.

    Outputs:
    --------
    preprocessor : sklearn.compose.ColumnTransformer
        Unfitted preprocessing object for use inside grouped model pipelines.

    Author:
    -------
    Rodrigo Kang
    """
    if feature_set_name not in feature_sets:
        raise KeyError(f"Unknown feature set: {feature_set_name}")

    selected_columns = feature_sets[feature_set_name]
    selected_set = set(selected_columns)

    categorical_columns = [
        column
        for column in management_columns
        if column in selected_set and raw_data[column].dtype == "object"
    ]
    conventional_numeric_columns = [
        column
        for column in management_columns + soil_chemistry_columns
        if column in selected_set and column not in categorical_columns
    ]
    microbial_numeric_columns = [
        column
        for column in qpcr_columns + shannon_columns + pcoa_columns
        if column in selected_set
    ]
    selected_bacterial_columns = [
        column for column in bacterial_taxa_columns if column in selected_set
    ]
    selected_fungal_columns = [
        column for column in fungal_taxa_columns if column in selected_set
    ]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    microbial_steps = [("imputer", SimpleImputer(strategy="median"))]
    taxa_steps = [("clr", PrevalenceCLRTransformer(min_prevalence=0.10))]

    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
        microbial_steps.append(("scaler", StandardScaler()))
        taxa_steps.append(("scaler", StandardScaler()))

    transformers = []

    if categorical_columns:
        categorical_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ),
            ]
        )
        transformers.append(("categorical", categorical_pipeline, categorical_columns))

    if conventional_numeric_columns:
        transformers.append(
            (
                "conventional-numeric",
                Pipeline(steps=numeric_steps),
                conventional_numeric_columns,
            )
        )

    if microbial_numeric_columns:
        transformers.append(
            (
                "microbial-numeric",
                Pipeline(steps=microbial_steps),
                microbial_numeric_columns,
            )
        )

    if selected_bacterial_columns:
        transformers.append(
            (
                "bacterial-taxa",
                Pipeline(steps=taxa_steps),
                selected_bacterial_columns,
            )
        )

    if selected_fungal_columns:
        transformers.append(
            (
                "fungal-taxa",
                Pipeline(steps=taxa_steps),
                selected_fungal_columns,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


x_data = raw_data.drop(columns=[target_column])
y_data = raw_data[target_column].copy()
groups = raw_data[group_column].copy()

outer_folds = 5
outer_cv = GroupKFold(n_splits=outer_folds)
outer_splits = list(outer_cv.split(x_data, y_data, groups=groups))

fold_assignments = pd.DataFrame(
    {
        "row-index": raw_data.index,
        "sample": raw_data["Sample"],
        "field-group": groups,
        "outer-fold": pd.Series(pd.NA, index=raw_data.index, dtype="Int64"),
    }
)

fold_summary_rows = []
for fold_number, (train_index, validation_index) in enumerate(outer_splits, start=1):
    train_groups = set(groups.iloc[train_index])
    validation_groups = set(groups.iloc[validation_index])
    overlap = train_groups.intersection(validation_groups)

    if overlap:
        raise RuntimeError(f"Field-group leakage detected in outer fold {fold_number}.")

    fold_assignments.loc[validation_index, "outer-fold"] = fold_number
    fold_summary_rows.append(
        {
            "outer-fold": fold_number,
            "training-observations": len(train_index),
            "validation-observations": len(validation_index),
            "training-groups": len(train_groups),
            "validation-groups": len(validation_groups),
            "group-overlap": len(overlap),
            "training-target-mean": y_data.iloc[train_index].mean(),
            "validation-target-mean": y_data.iloc[validation_index].mean(),
            "validation-target-sd": y_data.iloc[validation_index].std(),
        }
    )

fold_summary = pd.DataFrame(fold_summary_rows)

if fold_assignments["outer-fold"].isna().any():
    raise RuntimeError("At least one observation was not assigned to an outer fold.")

preprocessing_plan = pd.DataFrame(
    [
        {
            "feature-set": feature_set_name,
            "source-features": len(columns),
            "categorical-features": sum(
                column in management_columns and raw_data[column].dtype == "object"
                for column in columns
            ),
            "numeric-summary-features": sum(
                column in management_columns
                + soil_chemistry_columns
                + qpcr_columns
                + shannon_columns
                + pcoa_columns
                and raw_data[column].dtype != "object"
                for column in columns
            ),
            "bacterial-taxa": sum(column in bacterial_taxa_columns for column in columns),
            "fungal-taxa": sum(column in fungal_taxa_columns for column in columns),
            "uses-pcoa": any(column in pcoa_columns for column in columns),
            "uses-clr": any(
                column in bacterial_taxa_columns + fungal_taxa_columns
                for column in columns
            ),
        }
        for feature_set_name, columns in feature_sets.items()
    ]
)

# Fit one integrated preprocessor on the first training fold only as a structural test.
# This confirms that unseen rows can be transformed without fitting on validation data.
first_train_index, first_validation_index = outer_splits[0]
preprocessor_check = build_preprocessor("integrated", scale_numeric=True)
preprocessor_check.fit(x_data.iloc[first_train_index], y_data.iloc[first_train_index])
training_matrix_check = preprocessor_check.transform(x_data.iloc[first_train_index])
validation_matrix_check = preprocessor_check.transform(x_data.iloc[first_validation_index])

preprocessing_check = pd.DataFrame(
    {
        "partition": ["training", "validation"],
        "observations": [training_matrix_check.shape[0], validation_matrix_check.shape[0]],
        "transformed-features": [
            training_matrix_check.shape[1],
            validation_matrix_check.shape[1],
        ],
        "finite-values": [
            bool(np.isfinite(training_matrix_check).all()),
            bool(np.isfinite(validation_matrix_check).all()),
        ],
    }
)

if training_matrix_check.shape[1] != validation_matrix_check.shape[1]:
    raise RuntimeError("Training and validation preprocessing produced inconsistent columns.")
if not preprocessing_check["finite-values"].all():
    raise RuntimeError("Preprocessing produced missing or infinite values.")

save_table(fold_assignments, "outer-folds")
save_table(fold_summary, "outer-fold-summary")
save_table(preprocessing_plan, "preprocessing-plan")
save_table(preprocessing_check, "preprocessing-check")

display(fold_summary)
display(preprocessing_plan)
display(preprocessing_check)


,outer-fold,training-observations,validation-observations,training-groups,validation-groups,group-overlap,training-target-mean,validation-target-mean,validation-target-sd
0,1,338,85,62,15,0,"3,782.4567","3,744.9369","1,765.8689"
1,2,338,85,61,16,0,"3,777.4111","3,765.0005","1,306.3718"
2,3,339,84,62,15,0,"3,716.3178","4,011.4078","1,545.7559"
3,4,338,85,61,16,0,"3,779.7844","3,755.5634","1,472.8269"
4,5,339,84,62,15,0,"3,818.6603","3,598.3831","1,755.6822"


,feature-set,source-features,categorical-features,numeric-summary-features,bacterial-taxa,fungal-taxa,uses-pcoa,uses-clr
0,conventional,20,8,12,0,0,False,False
1,microbial-summaries,14,0,14,0,0,True,False
2,conventional-plus-summaries,34,8,26,0,0,True,False
3,taxonomic-composition,1591,0,0,885,706,False,True
4,conventional-plus-taxa,1611,8,12,885,706,False,True
5,integrated,1621,8,22,885,706,False,True


,partition,observations,transformed-features,finite-values
0,training,338,855,True
1,validation,85,855,True


## Model Development

This section defines a deliberately compact set of regression models representing distinct assumptions about the response surface.

The comparison includes a mean-prediction baseline, a regularised linear model, a bagged tree ensemble, and a boosted tree model. Hyperparameter grids are intentionally small because the scientific objective is to assess the incremental predictive value of microbial information, rather than to conduct a broad algorithm benchmark. All model selection will use three grouped inner folds nested within the grouped outer validation folds.

In [ ]:
def build_inner_group_splits(x, y, groups, n_splits=4):
    """
    Description:
    ------------
    Create deterministic grouped cross-validation splits for inner model selection.

    Input:
    ------
    x : pandas.DataFrame
        Training predictors available within one outer fold.
    y : pandas.Series
        Training target values available within one outer fold.
    groups : pandas.Series
        Field-group labels aligned with the training observations.
    n_splits : int
        Requested number of grouped inner folds.

    Outputs:
    --------
    inner_splits : list of tuple
        Training and validation index arrays for grouped inner cross-validation.

    Author:
    -------
    Rodrigo Kang
    """
    unique_groups = pd.Series(groups).nunique()
    effective_splits = min(n_splits, unique_groups)

    if effective_splits < 2:
        raise ValueError("At least two field groups are required for inner validation.")

    inner_cv = GroupKFold(n_splits=effective_splits)
    inner_splits = list(inner_cv.split(x, y, groups=groups))

    for split_number, (train_index, validation_index) in enumerate(inner_splits, start=1):
        train_groups = set(pd.Series(groups).iloc[train_index])
        validation_groups = set(pd.Series(groups).iloc[validation_index])

        if train_groups.intersection(validation_groups):
            raise RuntimeError(
                f"Field-group leakage detected in inner split {split_number}."
            )

    return inner_splits


def build_model_pipeline(feature_set_name, model_name):
    """
    Description:
    ------------
    Construct an unfitted preprocessing and regression pipeline for one model-feature pair.

    Input:
    ------
    feature_set_name : str
        Name of a predefined information block in feature_sets.
    model_name : str
        Name of a regression model defined in model_registry.

    Outputs:
    --------
    model_pipeline : sklearn.pipeline.Pipeline
        Unfitted leakage-safe modelling pipeline.

    Author:
    -------
    Rodrigo Kang
    """
    if model_name not in model_registry:
        raise KeyError(f"Unknown model: {model_name}")

    model_definition = model_registry[model_name]
    preprocessor = build_preprocessor(
        feature_set_name=feature_set_name,
        scale_numeric=model_definition["scale-numeric"],
    )

    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", clone(model_definition["estimator"])),
        ]
    )
    return model_pipeline


def build_model_search(feature_set_name, model_name, inner_splits):
    """
    Description:
    ------------
    Build a grouped grid-search object for one model and information block.

    Input:
    ------
    feature_set_name : str
        Name of a predefined information block in feature_sets.
    model_name : str
        Name of a regression model defined in model_registry.
    inner_splits : list of tuple
        Group-safe inner cross-validation indices created from outer-training data.

    Outputs:
    --------
    model_search : sklearn.model_selection.GridSearchCV or sklearn.pipeline.Pipeline
        Unfitted search object, or the baseline pipeline when no tuning is required.

    Author:
    -------
    Rodrigo Kang
    """
    model_pipeline = build_model_pipeline(feature_set_name, model_name)
    parameter_grid = model_registry[model_name]["parameter-grid"]

    if not parameter_grid:
        return model_pipeline

    model_search = GridSearchCV(
        estimator=model_pipeline,
        param_grid=parameter_grid,
        scoring="neg_mean_absolute_error",
        cv=inner_splits,
        n_jobs=4,
        pre_dispatch=4,
        refit=True,
        return_train_score=False,
        error_score="raise",
    )
    return model_search


model_registry = {
    "naive-mean": {
        "estimator": DummyRegressor(strategy="mean"),
        "scale-numeric": False,
        "parameter-grid": {},
        "rationale": "Reference performance from the training-fold mean yield.",
    },
    "elastic-net": {
        "estimator": ElasticNet(
            max_iter=20_000,
            random_state=random_seed,
        ),
        "scale-numeric": True,
        "parameter-grid": {
            "regressor__alpha": [0.10, 1.00],
            "regressor__l1_ratio": [0.20, 0.80],
        },
        "rationale": "Regularised additive effects in a potentially high-dimensional feature space.",
    },
    "random-forest": {
        "estimator": RandomForestRegressor(
            n_estimators=120,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=1,
            random_state=random_seed,
        ),
        "scale-numeric": False,
        "parameter-grid": {
            "regressor__max_depth": [None, 8],
            "regressor__min_samples_leaf": [3],
            "regressor__max_features": ["sqrt"],
        },
        "rationale": "Non-linear interactions and threshold effects with variance reduction by bagging.",
    },
    "gradient-boosting": {
        "estimator": GradientBoostingRegressor(
            loss="huber",
            random_state=random_seed,
        ),
        "scale-numeric": False,
        "parameter-grid": {
            "regressor__n_estimators": [100, 200],
            "regressor__learning_rate": [0.05],
            "regressor__max_depth": [1, 2],
            "regressor__min_samples_leaf": [5],
        },
        "rationale": "Sequential non-linear fitting with a robust loss and shallow component trees.",
    },
}

model_plan = pd.DataFrame(
    [
        {
            "model": model_name,
            "estimator": model_definition["estimator"].__class__.__name__,
            "scaled-inputs": model_definition["scale-numeric"],
            "tuned": bool(model_definition["parameter-grid"]),
            "candidate-settings": (
                int(np.prod([len(values) for values in model_definition["parameter-grid"].values()]))
                if model_definition["parameter-grid"]
                else 1
            ),
            "selection-metric": (
                "inner grouped MAE" if model_definition["parameter-grid"] else "not applicable"
            ),
            "rationale": model_definition["rationale"],
        }
        for model_name, model_definition in model_registry.items()
    ]
)

comparison_plan = pd.DataFrame(
    [
        {
            "feature-set": feature_set_name,
            "source-features": len(feature_sets[feature_set_name]),
            "models": ", ".join(model_registry.keys()),
            "outer-folds": outer_folds,
            "inner-folds": 3,
        }
        for feature_set_name in feature_sets
    ]
)

# Structural smoke test: construct every model-feature pipeline without fitting it.
pipeline_check_rows = []
for feature_set_name in feature_sets:
    for model_name in model_registry:
        candidate_pipeline = build_model_pipeline(feature_set_name, model_name)
        pipeline_check_rows.append(
            {
                "feature-set": feature_set_name,
                "model": model_name,
                "pipeline-steps": len(candidate_pipeline.steps),
                "constructed": True,
            }
        )

pipeline_check = pd.DataFrame(pipeline_check_rows)

# Validate grouped inner splits using only the first outer-training partition.
first_outer_train_index, _ = outer_splits[0]
inner_split_check = build_inner_group_splits(
    x=x_data.iloc[first_outer_train_index].reset_index(drop=True),
    y=y_data.iloc[first_outer_train_index].reset_index(drop=True),
    groups=groups.iloc[first_outer_train_index].reset_index(drop=True),
    n_splits=3,
)

inner_fold_summary = pd.DataFrame(
    [
        {
            "inner-fold": fold_number,
            "training-observations": len(train_index),
            "validation-observations": len(validation_index),
            "training-groups": groups.iloc[first_outer_train_index]
            .reset_index(drop=True)
            .iloc[train_index]
            .nunique(),
            "validation-groups": groups.iloc[first_outer_train_index]
            .reset_index(drop=True)
            .iloc[validation_index]
            .nunique(),
        }
        for fold_number, (train_index, validation_index) in enumerate(
            inner_split_check,
            start=1,
        )
    ]
)

save_table(model_plan, "model-plan")
save_table(comparison_plan, "comparison-plan")
save_table(pipeline_check, "pipeline-check")
save_table(inner_fold_summary, "inner-fold-summary")

display(model_plan)
display(comparison_plan)
display(inner_fold_summary)

## Training and Validation

This section performs nested grouped cross-validation. For each outer fold, hyperparameters are selected only from the corresponding training partition using three grouped inner folds. The fitted model then generates predictions for fields that were not used during preprocessing, feature filtering, tuning, or estimation.

The computation is checkpointed by information block, model, and outer fold. Each completed fold is written immediately under `output/tables/checkpoints/`, so an interrupted local run can be restarted without repeating finished fits. The final tables are consolidated automatically after all requested combinations are available.

The cell is intentionally left unexecuted in this distributed notebook because the complete nested evaluation is computationally intensive. Run the notebook from the local `python/` directory to generate the results.

In [ ]:
def calculate_regression_metrics(y_true, y_pred):
    """
    Description:
    ------------
    Calculate regression metrics for held-out predictions.

    Input:
    ------
    y_true : array-like
        Observed continuous target values.
    y_pred : array-like
        Predicted continuous target values.

    Outputs:
    --------
    metrics : dict
        Mean absolute error, root mean squared error, and R-squared.

    Author:
    -------
    Rodrigo Kang
    """
    y_true_array = np.asarray(y_true, dtype=float)
    y_pred_array = np.asarray(y_pred, dtype=float)

    if y_true_array.shape != y_pred_array.shape:
        raise ValueError("Observed and predicted values must have the same shape.")

    return {
        "mae": mean_absolute_error(y_true_array, y_pred_array),
        "rmse": np.sqrt(mean_squared_error(y_true_array, y_pred_array)),
        "r2": r2_score(y_true_array, y_pred_array),
    }


def checkpoint_paths(checkpoint_dir, feature_set_name, model_name, outer_fold):
    """
    Description:
    ------------
    Build deterministic paths for one completed outer-fold checkpoint.

    Input:
    ------
    checkpoint_dir : pathlib.Path
        Folder used to store intermediate validation tables.
    feature_set_name : str
        Name of the evaluated information block.
    model_name : str
        Name of the evaluated regression model.
    outer_fold : int
        One-based outer-fold number.

    Outputs:
    --------
    paths : dict
        Paths for metrics, predictions, and tuning results.

    Author:
    -------
    Rodrigo Kang
    """
    prefix = f"{feature_set_name}-{model_name}-fold-{outer_fold}"
    return {
        "metrics": checkpoint_dir / f"{prefix}-metrics.csv",
        "predictions": checkpoint_dir / f"{prefix}-predictions.csv",
        "tuning": checkpoint_dir / f"{prefix}-tuning.csv",
    }


def run_nested_grouped_validation(
    x,
    y,
    groups,
    sample_ids,
    feature_set_names,
    model_names,
    outer_splits,
    checkpoint_dir,
    inner_folds=3,
    overwrite=False,
):
    """
    Description:
    ------------
    Evaluate model and feature-set combinations using checkpointed nested grouped validation.

    Input:
    ------
    x : pandas.DataFrame
        Complete predictor table before fold-specific feature selection.
    y : pandas.Series
        Continuous regression target.
    groups : pandas.Series
        Physical-field grouping labels used to prevent leakage.
    sample_ids : pandas.Series
        Stable observation identifiers for exported predictions.
    feature_set_names : list of str
        Information blocks to evaluate.
    model_names : list of str
        Registered regression models to evaluate.
    outer_splits : list of tuple
        Fixed grouped training and validation indices.
    checkpoint_dir : pathlib.Path
        Folder in which completed outer-fold results are stored.
    inner_folds : int
        Number of grouped folds used for hyperparameter selection.
    overwrite : bool
        Whether to recompute checkpoints that already exist.

    Outputs:
    --------
    validation_outputs : dict
        Consolidated fold metrics, out-of-fold predictions, and tuning results.

    Author:
    -------
    Rodrigo Kang
    """
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    for feature_set_name in feature_set_names:
        x_feature_set = x.loc[:, feature_sets[feature_set_name]]

        for model_name in model_names:
            for outer_fold, (train_index, validation_index) in enumerate(
                outer_splits,
                start=1,
            ):
                paths = checkpoint_paths(
                    checkpoint_dir=checkpoint_dir,
                    feature_set_name=feature_set_name,
                    model_name=model_name,
                    outer_fold=outer_fold,
                )

                checkpoint_complete = all(path.exists() for path in paths.values())
                if checkpoint_complete and not overwrite:
                    print(
                        f"Skipping completed checkpoint: "
                        f"{feature_set_name} | {model_name} | fold {outer_fold}"
                    )
                    continue

                print(
                    f"Fitting: {feature_set_name} | {model_name} | "
                    f"outer fold {outer_fold}/{len(outer_splits)}"
                )

                x_train = x_feature_set.iloc[train_index].reset_index(drop=True)
                x_validation = x_feature_set.iloc[validation_index].reset_index(drop=True)
                y_train = y.iloc[train_index].reset_index(drop=True)
                y_validation = y.iloc[validation_index].reset_index(drop=True)
                groups_train = groups.iloc[train_index].reset_index(drop=True)

                inner_splits = build_inner_group_splits(
                    x=x_train,
                    y=y_train,
                    groups=groups_train,
                    n_splits=inner_folds,
                )
                fitted_search = build_model_search(
                    feature_set_name=feature_set_name,
                    model_name=model_name,
                    inner_splits=inner_splits,
                )
                fitted_search.fit(x_train, y_train)
                validation_predictions = fitted_search.predict(x_validation)

                metrics = calculate_regression_metrics(
                    y_true=y_validation,
                    y_pred=validation_predictions,
                )
                fold_table = pd.DataFrame(
                    [
                        {
                            "feature-set": feature_set_name,
                            "model": model_name,
                            "outer-fold": outer_fold,
                            "validation-observations": len(validation_index),
                            "validation-groups": groups.iloc[validation_index].nunique(),
                            **metrics,
                        }
                    ]
                )

                if hasattr(fitted_search, "best_params_"):
                    best_parameters = fitted_search.best_params_
                    inner_mae = -float(fitted_search.best_score_)
                else:
                    best_parameters = {}
                    inner_mae = np.nan

                tuning_table = pd.DataFrame(
                    [
                        {
                            "feature-set": feature_set_name,
                            "model": model_name,
                            "outer-fold": outer_fold,
                            "inner-mae": inner_mae,
                            "best-parameters": str(best_parameters),
                        }
                    ]
                )

                prediction_table = pd.DataFrame(
                    {
                        "sample": sample_ids.iloc[validation_index].to_numpy(),
                        "field-group": groups.iloc[validation_index].to_numpy(),
                        "feature-set": feature_set_name,
                        "model": model_name,
                        "outer-fold": outer_fold,
                        "observed-yield": y_validation.to_numpy(dtype=float),
                        "predicted-yield": np.asarray(
                            validation_predictions,
                            dtype=float,
                        ),
                    }
                )
                prediction_table["residual"] = (
                    prediction_table["observed-yield"]
                    - prediction_table["predicted-yield"]
                )

                fold_table.to_csv(paths["metrics"], index=False)
                prediction_table.to_csv(paths["predictions"], index=False)
                tuning_table.to_csv(paths["tuning"], index=False)

    metric_files = sorted(checkpoint_dir.glob("*-metrics.csv"))
    prediction_files = sorted(checkpoint_dir.glob("*-predictions.csv"))
    tuning_files = sorted(checkpoint_dir.glob("*-tuning.csv"))

    expected_checkpoints = (
        len(feature_set_names) * len(model_names) * len(outer_splits)
    )
    if not (
        len(metric_files) == expected_checkpoints
        and len(prediction_files) == expected_checkpoints
        and len(tuning_files) == expected_checkpoints
    ):
        raise RuntimeError(
            "Validation is incomplete. Re-run this cell to resume from the "
            "available checkpoints."
        )

    return {
        "fold-metrics": pd.concat(
            [pd.read_csv(path) for path in metric_files],
            ignore_index=True,
        ),
        "predictions": pd.concat(
            [pd.read_csv(path) for path in prediction_files],
            ignore_index=True,
        ),
        "tuning-results": pd.concat(
            [pd.read_csv(path) for path in tuning_files],
            ignore_index=True,
        ),
    }


checkpoint_dir = tables_dir / "checkpoints"

validation_outputs = run_nested_grouped_validation(
    x=x_data,
    y=y_data,
    groups=groups,
    sample_ids=raw_data["Sample"],
    feature_set_names=list(feature_sets),
    model_names=list(model_registry),
    outer_splits=outer_splits,
    checkpoint_dir=checkpoint_dir,
    inner_folds=3,
    overwrite=False,
)

fold_metrics = validation_outputs["fold-metrics"]
out_of_fold_predictions = validation_outputs["predictions"]
tuning_results = validation_outputs["tuning-results"]

expected_predictions = len(raw_data) * len(feature_sets) * len(model_registry)
if len(out_of_fold_predictions) != expected_predictions:
    raise RuntimeError(
        "Expected one out-of-fold prediction per observation, feature set, and model."
    )

prediction_counts = (
    out_of_fold_predictions.groupby(["feature-set", "model"], as_index=False)
    .agg(
        **{
            "observations": ("sample", "size"),
            "unique-samples": ("sample", "nunique"),
            "folds": ("outer-fold", "nunique"),
        }
    )
)

if not (
    prediction_counts["observations"].eq(len(raw_data)).all()
    and prediction_counts["unique-samples"].eq(len(raw_data)).all()
    and prediction_counts["folds"].eq(outer_folds).all()
):
    raise RuntimeError("Out-of-fold prediction coverage is incomplete or duplicated.")

save_table(fold_metrics, "fold-metrics")
save_table(out_of_fold_predictions, "oof-predictions")
save_table(tuning_results, "tuning-results")
save_table(prediction_counts, "prediction-check")

display(fold_metrics.head(10))
display(prediction_counts)

## Results and Evaluation

This section reads the consolidated out-of-fold results created during local training and summarises predictive performance across the fixed grouped outer folds.

The primary comparison is whether microbial information improves prediction beyond the `conventional` block. Results are therefore reported both as absolute cross-validated error and as paired fold-level changes relative to that reference. Lower MAE and RMSE indicate better predictions, while higher R² is preferable. Fold-level standard deviations are retained to show uncertainty arising from the limited number of commercial fields.

The code is intentionally independent of the training cell. If the consolidated CSV files do not yet exist, it reports which files are missing rather than failing the remainder of the notebook.

In [ ]:
def load_validation_outputs(tables_directory):
    """
    Description:
    ------------
    Load consolidated cross-validation outputs when local training has completed.

    Input:
    ------
    tables_directory : pathlib.Path
        Folder containing the exported validation CSV files.

    Outputs:
    --------
    validation_tables : dict or None
        Loaded fold metrics, out-of-fold predictions, and tuning results, or None
        when one or more required files are unavailable.

    Author:
    -------
    Rodrigo Kang
    """
    required_files = {
        "fold-metrics": tables_directory / "fold-metrics.csv",
        "oof-predictions": tables_directory / "oof-predictions.csv",
        "tuning-results": tables_directory / "tuning-results.csv",
    }
    missing_files = [path.name for path in required_files.values() if not path.exists()]

    if missing_files:
        print(
            "Results are not available yet. Run the Training and Validation cell "
            "locally to create: " + ", ".join(missing_files)
        )
        return None

    return {
        table_name: pd.read_csv(table_path)
        for table_name, table_path in required_files.items()
    }


def summarise_fold_metrics(fold_metrics_table):
    """
    Description:
    ------------
    Summarise grouped outer-fold regression metrics by information block and model.

    Input:
    ------
    fold_metrics_table : pandas.DataFrame
        Fold-level MAE, RMSE, and R-squared values.

    Outputs:
    --------
    metric_summary : pandas.DataFrame
        Mean, standard deviation, minimum, and maximum metrics across outer folds.

    Author:
    -------
    Rodrigo Kang
    """
    required_columns = {"feature-set", "model", "outer-fold", "mae", "rmse", "r2"}
    missing_columns = required_columns.difference(fold_metrics_table.columns)
    if missing_columns:
        raise ValueError(f"Fold metrics are missing columns: {sorted(missing_columns)}")

    metric_summary = (
        fold_metrics_table.groupby(["feature-set", "model"], as_index=False)
        .agg(
            **{
                "folds": ("outer-fold", "nunique"),
                "mae-mean": ("mae", "mean"),
                "mae-std": ("mae", "std"),
                "rmse-mean": ("rmse", "mean"),
                "rmse-std": ("rmse", "std"),
                "r2-mean": ("r2", "mean"),
                "r2-std": ("r2", "std"),
                "r2-min": ("r2", "min"),
                "r2-max": ("r2", "max"),
            }
        )
        .sort_values(["mae-mean", "rmse-mean", "r2-mean"], ascending=[True, True, False])
        .reset_index(drop=True)
    )
    return metric_summary


def calculate_incremental_value(fold_metrics_table, reference_feature_set="conventional"):
    """
    Description:
    ------------
    Calculate paired fold-level performance changes relative to a reference block.

    Input:
    ------
    fold_metrics_table : pandas.DataFrame
        Fold-level validation metrics for all model and information-block combinations.
    reference_feature_set : str
        Information block used as the conventional-input reference.

    Outputs:
    --------
    incremental_summary : pandas.DataFrame
        Mean and variability of paired MAE, RMSE, and R-squared changes by model.

    Author:
    -------
    Rodrigo Kang
    """
    reference_metrics = (
        fold_metrics_table.loc[
            fold_metrics_table["feature-set"].eq(reference_feature_set),
            ["model", "outer-fold", "mae", "rmse", "r2"],
        ]
        .rename(
            columns={
                "mae": "reference-mae",
                "rmse": "reference-rmse",
                "r2": "reference-r2",
            }
        )
    )

    paired_metrics = fold_metrics_table.merge(
        reference_metrics,
        on=["model", "outer-fold"],
        how="inner",
        validate="many_to_one",
    )
    paired_metrics = paired_metrics.loc[
        ~paired_metrics["feature-set"].eq(reference_feature_set)
    ].copy()
    paired_metrics["mae-improvement"] = (
        paired_metrics["reference-mae"] - paired_metrics["mae"]
    )
    paired_metrics["rmse-improvement"] = (
        paired_metrics["reference-rmse"] - paired_metrics["rmse"]
    )
    paired_metrics["r2-improvement"] = (
        paired_metrics["r2"] - paired_metrics["reference-r2"]
    )

    incremental_summary = (
        paired_metrics.groupby(["feature-set", "model"], as_index=False)
        .agg(
            **{
                "folds": ("outer-fold", "nunique"),
                "mae-improvement-mean": ("mae-improvement", "mean"),
                "mae-improvement-std": ("mae-improvement", "std"),
                "rmse-improvement-mean": ("rmse-improvement", "mean"),
                "rmse-improvement-std": ("rmse-improvement", "std"),
                "r2-improvement-mean": ("r2-improvement", "mean"),
                "r2-improvement-std": ("r2-improvement", "std"),
                "mae-better-folds": ("mae-improvement", lambda values: int((values > 0).sum())),
            }
        )
        .sort_values("mae-improvement-mean", ascending=False)
        .reset_index(drop=True)
    )
    return incremental_summary


def select_best_combinations(metric_summary):
    """
    Description:
    ------------
    Identify the lowest-MAE model within each information block and overall.

    Input:
    ------
    metric_summary : pandas.DataFrame
        Aggregated cross-validation metrics by information block and model.

    Outputs:
    --------
    best_combinations : pandas.DataFrame
        Best model for each information block, with the overall best row flagged.

    Author:
    -------
    Rodrigo Kang
    """
    best_combinations = (
        metric_summary.sort_values(["feature-set", "mae-mean"])
        .groupby("feature-set", as_index=False)
        .first()
        .sort_values("mae-mean")
        .reset_index(drop=True)
    )
    best_combinations["overall-best"] = False
    if not best_combinations.empty:
        best_combinations.loc[0, "overall-best"] = True
    return best_combinations


def plot_cross_validated_mae(metric_summary):
    """
    Description:
    ------------
    Plot mean grouped-cross-validation MAE with fold-level standard deviations.

    Input:
    ------
    metric_summary : pandas.DataFrame
        Aggregated validation metrics by information block and model.

    Outputs:
    --------
    figure : matplotlib.figure.Figure
        Figure comparing predictive error across evaluated combinations.

    Author:
    -------
    Rodrigo Kang
    """
    plot_table = metric_summary.copy()
    plot_table["combination"] = (
        plot_table["feature-set"] + " | " + plot_table["model"]
    )
    plot_table = plot_table.sort_values("mae-mean", ascending=True)

    figure_height = max(6, 0.32 * len(plot_table))
    figure, axis = plt.subplots(figsize=(10, figure_height))
    positions = np.arange(len(plot_table))
    axis.barh(
        positions,
        plot_table["mae-mean"],
        xerr=plot_table["mae-std"].fillna(0),
        capsize=3,
    )
    axis.set_yticks(positions)
    axis.set_yticklabels(plot_table["combination"])
    axis.set_xlabel("Mean absolute error (grams per metre)")
    axis.set_ylabel("")
    axis.set_title("Grouped cross-validated prediction error")
    return figure


def plot_observed_vs_predicted(prediction_table, feature_set_name, model_name):
    """
    Description:
    ------------
    Plot out-of-fold observed and predicted yields for one selected combination.

    Input:
    ------
    prediction_table : pandas.DataFrame
        Out-of-fold predictions for all evaluated combinations.
    feature_set_name : str
        Information block to display.
    model_name : str
        Regression model to display.

    Outputs:
    --------
    figure : matplotlib.figure.Figure
        Observed-versus-predicted diagnostic plot.

    Author:
    -------
    Rodrigo Kang
    """
    selected = prediction_table.loc[
        prediction_table["feature-set"].eq(feature_set_name)
        & prediction_table["model"].eq(model_name)
    ].copy()
    if selected.empty:
        raise ValueError("No out-of-fold predictions match the selected combination.")

    lower_limit = min(selected["observed-yield"].min(), selected["predicted-yield"].min())
    upper_limit = max(selected["observed-yield"].max(), selected["predicted-yield"].max())

    figure, axis = plt.subplots(figsize=(7, 6))
    axis.scatter(selected["observed-yield"], selected["predicted-yield"], alpha=0.65)
    axis.plot([lower_limit, upper_limit], [lower_limit, upper_limit], linestyle="--")
    axis.set_xlim(lower_limit, upper_limit)
    axis.set_ylim(lower_limit, upper_limit)
    axis.set_xlabel("Observed yield (grams per metre)")
    axis.set_ylabel("Out-of-fold predicted yield (grams per metre)")
    axis.set_title(f"Observed versus predicted: {feature_set_name} | {model_name}")
    return figure


validation_tables = load_validation_outputs(tables_dir)

if validation_tables is not None:
    fold_metrics = validation_tables["fold-metrics"]
    out_of_fold_predictions = validation_tables["oof-predictions"]
    tuning_results = validation_tables["tuning-results"]

    cv_summary = summarise_fold_metrics(fold_metrics)
    incremental_value = calculate_incremental_value(fold_metrics)
    best_combinations = select_best_combinations(cv_summary)

    save_table(cv_summary, "cv-summary")
    save_table(incremental_value, "incremental-value")
    save_table(best_combinations, "best-combinations")

    mae_figure = plot_cross_validated_mae(cv_summary)
    save_figure(mae_figure, "cv-mae")

    overall_best = best_combinations.loc[best_combinations["overall-best"]].iloc[0]
    observed_figure = plot_observed_vs_predicted(
        prediction_table=out_of_fold_predictions,
        feature_set_name=overall_best["feature-set"],
        model_name=overall_best["model"],
    )
    save_figure(observed_figure, "observed-predicted")

    display(cv_summary)
    display(incremental_value)
    display(best_combinations)


## Additional Experiments (Optional)


This section is reserved for targeted robustness checks after the primary analysis has been completed. Suitable checks include an alternative field-group definition, sensitivity to the taxon-prevalence threshold, and summaries by project or region where sample sizes are sufficient.

These checks should remain secondary. They are not used to search broadly for a favourable result, and they should not replace the pre-specified comparison between conventional information and microbial feature blocks.


## Discussion


The discussion is generated from the grouped out-of-fold results rather than from training-set fit. It identifies the best-performing model and information block, quantifies the paired change relative to the conventional reference, and separates predictive evidence from biological interpretation.

The central question is not whether microbial variables are associated with yield in isolation. It is whether they provide reproducible predictive information beyond management and soil chemistry when entire commercial fields are held out.


In [ ]:
def build_discussion_summary(metric_summary, incremental_summary):
    """
    Description:
    ------------
    Build a compact evidence table for the main scientific comparison.

    Input:
    ------
    metric_summary : pandas.DataFrame
        Cross-validated metric summary by information block and model.
    incremental_summary : pandas.DataFrame
        Paired performance changes relative to the conventional feature block.

    Outputs:
    --------
    discussion_summary : pandas.DataFrame
        Best overall combination and strongest incremental comparison.

    Author:
    -------
    Rodrigo Kang
    """
    if metric_summary.empty:
        return pd.DataFrame()

    best_overall = metric_summary.sort_values(
        ["mae-mean", "rmse-mean", "r2-mean"],
        ascending=[True, True, False],
    ).iloc[0]

    summary_rows = [
        {
            "comparison": "best-overall",
            "feature-set": best_overall["feature-set"],
            "model": best_overall["model"],
            "mae": best_overall["mae-mean"],
            "rmse": best_overall["rmse-mean"],
            "r2": best_overall["r2-mean"],
            "mae-change-vs-conventional": np.nan,
            "improved-folds": np.nan,
        }
    ]

    if not incremental_summary.empty:
        best_increment = incremental_summary.sort_values(
            ["mae-change-mean", "rmse-change-mean", "r2-change-mean"],
            ascending=[True, True, False],
        ).iloc[0]
        summary_rows.append(
            {
                "comparison": "strongest-incremental",
                "feature-set": best_increment["feature-set"],
                "model": best_increment["model"],
                "mae": np.nan,
                "rmse": np.nan,
                "r2": np.nan,
                "mae-change-vs-conventional": best_increment["mae-change-mean"],
                "improved-folds": best_increment["mae-improved-folds"],
            }
        )

    return pd.DataFrame(summary_rows)


def format_predictive_conclusion(discussion_summary):
    """
    Description:
    ------------
    Convert the evidence summary into a cautious result-dependent conclusion.

    Input:
    ------
    discussion_summary : pandas.DataFrame
        Compact table containing the best overall and incremental comparisons.

    Outputs:
    --------
    conclusion_text : str
        Portfolio-ready interpretation of the grouped validation results.

    Author:
    -------
    Rodrigo Kang
    """
    if discussion_summary.empty:
        return (
            "The predictive conclusion is intentionally deferred until the local "
            "nested grouped validation has produced the consolidated result tables."
        )

    best_overall = discussion_summary.loc[
        discussion_summary["comparison"].eq("best-overall")
    ].iloc[0]
    incremental_rows = discussion_summary.loc[
        discussion_summary["comparison"].eq("strongest-incremental")
    ]

    opening = (
        f"The lowest mean grouped-CV MAE was obtained by {best_overall['model']} "
        f"using the {best_overall['feature-set']} feature set "
        f"(MAE {best_overall['mae']:.2f}, RMSE {best_overall['rmse']:.2f}, "
        f"R² {best_overall['r2']:.3f})."
    )

    if incremental_rows.empty:
        return opening

    incremental = incremental_rows.iloc[0]
    mae_change = incremental["mae-change-vs-conventional"]
    improved_folds = int(incremental["improved-folds"])
    direction = "lower" if mae_change < 0 else "higher"
    evidence = "supports" if mae_change < 0 and improved_folds >= 3 else "does not clearly support"

    return (
        opening
        + " "
        + f"The strongest microbial extension produced a {direction} paired MAE "
        f"by {abs(mae_change):.2f} grams per metre and improved {improved_folds} "
        f"of the five held-out folds. On this basis, the analysis {evidence} the "
        "claim that pre-plant microbial information adds stable predictive value "
        "beyond conventional management and soil chemistry variables."
    )


if validation_outputs is None:
    discussion_summary = pd.DataFrame()
    predictive_conclusion = format_predictive_conclusion(discussion_summary)
    print(predictive_conclusion)
else:
    discussion_summary = build_discussion_summary(
        metric_summary=cv_summary,
        incremental_summary=incremental_value,
    )
    save_table(discussion_summary, "discussion-summary")
    predictive_conclusion = format_predictive_conclusion(discussion_summary)
    display(discussion_summary)
    print(predictive_conclusion)


### Interpretation Boundaries


Any improvement should be interpreted as incremental predictive value under the chosen validation design. It does not establish that the selected microbial taxa, diversity measures, or abundance summaries cause changes in potato yield.

Feature effects can reflect correlated environmental conditions, management history, geography, season, measurement choices, or unobserved field characteristics. Conversely, a weak incremental result would not prove that soil microbial communities are biologically irrelevant; it would indicate that the available pre-plant measurements did not add stable out-of-field predictive information under this sample size and modelling strategy.


## Limitations


Several limitations constrain the strength and generalisability of the conclusions:

- The dataset is observational, so predictive associations cannot be treated as causal biological mechanisms.
- The published description refers to 130 commercial fields, while the available identifiers do not reproduce that count exactly. The conservative `State + Field1` grouping reduces leakage risk but may merge field-season records that could also be analysed separately.
- Only 423 observations are available, and the effective validation sample size is closer to the number of independent fields than to the number of rows.
- Taxonomic matrices are sparse, high-dimensional and truncated to selected genera. CLR transformation addresses relative scale but cannot recover unreported taxa or eliminate all compositional uncertainty.
- Management, chemistry and microbiology may vary by region, year, cultivar and laboratory workflow. Performance in new production systems may therefore differ from grouped cross-validation within this dataset.
- Hyperparameter grids are intentionally compact. This supports reproducibility and limits overfitting, but it does not guarantee the globally best configuration for each algorithm.
- The associated manuscript was under peer review when the dataset was posted, so biological conclusions should remain provisional and should be checked against the final publication when available.


## Conclusion


This project evaluates a focused practical question: whether pre-plant soil microbial measurements improve potato-yield prediction beyond management and soil chemistry variables.

The notebook uses conservative field-level grouping, nested cross-validation, fold-specific preprocessing, and incremental information blocks to make that comparison without assuming row-level independence. The final predictive statement is produced from the local out-of-fold results by the Discussion cell above. Regardless of the direction of the result, the appropriate conclusion concerns reproducible predictive value in held-out fields—not causal attribution to individual microbial features.


## Final Remarks


The analysis is designed to remain scientifically modest and operationally reproducible. It prioritises a defensible validation unit, transparent feature blocks, a compact model set, and exportable results over an expansive algorithm benchmark.

After running the local training cell, rerun **Results and Evaluation** and **Discussion** to populate all final tables, figures and result-dependent narrative. The resulting notebook can then be rendered or used as the computational companion to the Quarto portfolio chapter.
